In [ ]:
# ============================
# STEP 1: SETUP & IMPORTS
# ============================

# 1️⃣ Clone Ultralytics (YOLOv8/YOLO11) repo
!git clone https://github.com/ultralytics/ultralytics
%cd /content/ultralytics

# 2️⃣ Install in editable mode so our code changes take effect
!pip install -e .

# 3️⃣ All imports we will need later (import everything in Step 1)
import os
import re
import glob
import shutil
from pathlib import Path

import torch
import yaml

print("✅ Environment ready!")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("⚠ No GPU detected.")

Cloning into 'ultralytics'...
remote: Enumerating objects: 76173, done.
remote: Counting objects: 100% (178/178), done.
remote: Compressing objects: 100% (121/121), done.
remote: Total 76173 (delta 90), reused 97 (delta 57), pack-reused 75995 (from 2)
Receiving objects: 100% (76173/76173), 40.60 MiB | 13.32 MiB/s, done.
Resolving deltas: 100% (57234/57234), done.
/content/ultralytics
Obtaining file:///content/ultralytics
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ultralytics (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.3.233-0.editable-py3-none-any.whl size=23167 sha256=bc32a3109be65508e13e487e9cf6d4bdd8e1dbf860a7ab7d5f89a4c069dd342f
  Stored in directory: /tmp/pip-ephem-wheel-cache-wtkyjn9a/wheels/60/e0/59/e2f034f296abbdca5c21e3f5be76b9ca685f13c7bd17f8b58c
Succes

In [ ]:
# STEP 2A: Configure Kaggle API (run this once)

from google.colab import files
import os

print("📁 Please upload your kaggle.json file (from Kaggle account settings).")
uploaded = files.upload()  # <-- You select kaggle.json here

if 'kaggle.json' not in uploaded:
    raise FileNotFoundError("❌ kaggle.json not uploaded. Please try again.")

# Create kaggle folder and move the file
os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/

# Set proper permissions so Kaggle CLI accepts it
!chmod 600 /root/.kaggle/kaggle.json

print("✅ Kaggle API configured successfully!")

📁 Please upload your kaggle.json file (from Kaggle account settings).


Saving kaggle.json to kaggle.json
✅ Kaggle API configured successfully!


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class EPSANetAttention(nn.Module):
    def __init__(self, in_channels, k_size=3):
        super(EPSANetAttention, self).__init__()
        self.in_channels = in_channels
        self.k_size = k_size

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k_size, padding=(k_size - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

        print(f"Initialized EPSANetAttention with in_channels={in_channels}, k_size={k_size}")

    def forward(self, x):
        # x: input features with shape (batch_size, in_channels, H, W)

        # Squeeze operation: global average pooling along spatial dimensions
        # (batch_size, in_channels, H, W) -> (batch_size, in_channels, 1, 1)
        y = self.avg_pool(x)

        # Permute to (batch_size, 1, in_channels) for 1D convolution
        y = y.squeeze(-1).permute(0, 2, 1)

        # Excitation operation: 1D convolution
        # (batch_size, 1, in_channels) -> (batch_size, 1, in_channels)
        y = self.conv(y)

        # Permute back and unsqueeze for broadcasting
        # (batch_size, 1, in_channels) -> (batch_size, in_channels, 1, 1)
        y = y.permute(0, 2, 1).unsqueeze(-1)

        # Apply sigmoid to get attention weights
        attention_weights = self.sigmoid(y)

        # Scale the input features
        out = x * attention_weights

        return out

print("✅ EPSANetAttention class defined.")

✅ EPSANetAttention class defined.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from ultralytics.nn.modules import Conv, C2f # Assuming Conv, C2f are available

# --- Utility Block: SEWeight Module (Squeeze-and-Excitation) ---
# Used to generate channel attention for each scale in PSA
class SEWeightModule(nn.Module):
    def __init__(self, channels, reduction=16):
        super(SEWeightModule, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(channels, channels // reduction, kernel_size=1, padding=0)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Conv2d(channels // reduction, channels, kernel_size=1, padding=0)

    def forward(self, x):
        out = self.avg_pool(x)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        return out

# --- The EPSA Attention Module (PSA) ---
class PSA_Attention(nn.Module):
    """
    Pyramid Split Attention (PSA) module for EPSANet.
    It integrates multi-scale features and re-calibrates channel attention.
    """
    def __init__(self, in_channels, scales=4, reduction=16):
        super(PSA_Attention, self).__init__()
        self.scales = scales
        c_per_scale = in_channels // scales
        self.c_per_scale = c_per_scale

        # 1. Multi-scale feature extraction (Split and Concat - SPC)
        # We use a depthwise grouped convolution with different dilation rates
        # for simplicity and multi-scale feature extraction.
        # This acts as the 'Pyramid' part, capturing local and global info.
        self.conv_groups = nn.ModuleList()
        # Common dilations are (1, 2, 3, 4) for 4 scales
        dilations = [1, 2, 3, 4][:scales]

        for i in range(scales):
            # Each branch processes a part of the channels (c_per_scale)
            # using a 3x3 depthwise convolution with a different dilation.
            conv = nn.Conv2d(
                c_per_scale,
                c_per_scale,
                kernel_size=3,
                padding=dilations[i],
                dilation=dilations[i],
                groups=c_per_scale,
                bias=False
            )
            self.conv_groups.append(conv)

        # 2. SEWeight for each scale
        self.se_weights = nn.ModuleList(
            SEWeightModule(c_per_scale, reduction) for _ in range(scales)
        )

        # Final 1x1 convolution to mix the features after attention
        self.conv_fuse = Conv(in_channels, in_channels, 1, 1)

    def forward(self, x):
        # 1. Channel Splitting
        chunks = torch.split(x, self.c_per_scale, dim=1) # Split channels into 'scales' chunks

        # 2. Multi-scale convolution and Squeeze-and-Excitation
        features = []
        attention_vectors = []
        for i in range(self.scales):
            f_i = self.conv_groups[i](chunks[i])
            features.append(f_i)
            # SE on each scale feature
            w_i = self.se_weights[i](f_i)
            attention_vectors.append(w_i)

        # 3. Concatenate and Softmax Re-calibration (The 'A' in PSA)
        # Concatenate attention vectors along the channel dimension
        attention_cat = torch.cat(attention_vectors, dim=1)

        # Apply Softmax across the scales/groups (channel dimension)
        # Reshape to (B, scales, C_per_scale, 1, 1) for Softmax
        B, C, H, W = attention_cat.shape
        attention_reshaped = attention_cat.reshape(B, self.scales, self.c_per_scale, 1, 1)
        # Softmax applies across the 'scales' dimension (dim=1)
        softmax_attention = F.softmax(attention_reshaped, dim=1)
        attention_recalibrated = softmax_attention.reshape(B, C, 1, 1)

        # 4. Re-weighting and Fusion
        reweighted_features = []
        for i in range(self.scales):
            # Element-wise multiplication: Feature * Softmax Attention Weight
            # The attention for scale 'i' is the i-th slice of the recalibrated tensor
            attention_i = attention_recalibrated.narrow(1, i * self.c_per_scale, self.c_per_scale)
            reweighted_features.append(features[i] * attention_i.expand_as(features[i]))

        # Concatenate re-weighted features and final 1x1 Conv fusion
        out = torch.cat(reweighted_features, dim=1)
        out = self.conv_fuse(out)

        return out

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
class C2f_EPSA(nn.Module):
    """C2f block with Efficient Pyramid Squeeze Attention (EPSA/PSA) module."""
    # Add scales and reduction parameters for the PSA_Attention module
    def __init__(self, c1, c2, n=1, shortcut=False, g=1, e=0.5, scales=4, reduction=16):
        super().__init__()
        self.c = int(c2 * e)  # hidden channels
        self.cv1 = Conv(c1, 2 * self.c, 1, 1)
        self.cv2 = Conv((2 + n) * self.c, c2, 1)
        self.m = nn.ModuleList(
            nn.Sequential(
                Conv(self.c, self.c, 3, 1, g=g),
                Conv(self.c, self.c, 3, 1, g=g)
            ) for _ in range(n)
        )
        # 🟢 THE EPSA ATTENTION INJECTION 🟢
        # The PSA module replaces the final 1x1 Conv and feature multiplication
        # often found in attention blocks. It takes the output channel c2.
        self.att = PSA_Attention(c2, scales=scales, reduction=reduction)

    def forward(self, x):
        # YOLOv8 C2f Logic
        # Split the output of the first Conv into two parts: a direct path and a path through 'n' bottles
        y = list(self.cv1(x).chunk(2, 1))
        # Pass the last chunk through the 'n' repeated Conv blocks
        y.extend(m(y[-1]) for m in self.m)
        # Concatenate all parts and pass through the final 1x1 Conv (cv2)
        out = self.cv2(torch.cat(y, 1))

        # Apply EPSA Attention on the output features
        return self.att(out)

print("✅ PSA_Attention module defined, and C2f_EPSA block created, integrating EPSA.")

✅ PSA_Attention module defined, and C2f_EPSA block created, integrating EPSA.


In [ ]:
# --- 1. EPSA Attention (PSA) Module Definition ---
import torch
import torch.nn as nn
import torch.nn.functional as F
from ultralytics.nn.modules import Conv, C2f # Assuming Conv, C2f are available
from ultralytics import YOLO # Import the YOLO class

class SEWeightModule(nn.Module):
    def __init__(self, channels, reduction=16):
        super(SEWeightModule, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(channels, channels // reduction, kernel_size=1, padding=0)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Conv2d(channels // reduction, channels, kernel_size=1, padding=0)

    def forward(self, x):
        out = self.avg_pool(x)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        return out

class PSA_Attention(nn.Module):
    """Pyramid Split Attention (PSA) module for EPSANet."""
    def __init__(self, in_channels, scales=4, reduction=16):
        super(PSA_Attention, self).__init__()
        self.scales = scales
        c_per_scale = in_channels // scales
        self.c_per_scale = c_per_scale

        # Multi-scale feature extraction (Dilated Depthwise Convs)
        self.conv_groups = nn.ModuleList()
        dilations = [1, 2, 3, 4][:scales]

        for i in range(scales):
            conv = nn.Conv2d(
                c_per_scale,
                c_per_scale,
                kernel_size=3,
                padding=dilations[i],
                dilation=dilations[i],
                groups=c_per_scale,
                bias=False
            )
            self.conv_groups.append(conv)

        # SEWeight for each scale
        self.se_weights = nn.ModuleList(
            SEWeightModule(c_per_scale, reduction) for _ in range(scales)
        )

        # Final 1x1 convolution to fuse the features
        self.conv_fuse = Conv(in_channels, in_channels, 1, 1)

    def forward(self, x):
        # 1. Channel Splitting
        chunks = torch.split(x, self.c_per_scale, dim=1)

        # 2. Multi-scale convolution and Squeeze-and-Excitation
        features, attention_vectors = [], []
        for i in range(self.scales):
            f_i = self.conv_groups[i](chunks[i])
            features.append(f_i)
            w_i = self.se_weights[i](f_i)
            attention_vectors.append(w_i)

        # 3. Concatenate and Softmax Re-calibration
        attention_cat = torch.cat(attention_vectors, dim=1)

        # Softmax across the scales dimension
        B, C, H, W = attention_cat.shape
        attention_reshaped = attention_cat.reshape(B, self.scales, self.c_per_scale, 1, 1)
        softmax_attention = F.softmax(attention_reshaped, dim=1)
        attention_recalibrated = softmax_attention.reshape(B, C, 1, 1)

        # 4. Re-weighting and Fusion
        reweighted_features = []
        for i in range(self.scales):
            attention_i = attention_recalibrated.narrow(1, i * self.c_per_scale, self.c_per_scale)
            reweighted_features.append(features[i] * attention_i.expand_as(features[i]))

        out = torch.cat(reweighted_features, dim=1)
        out = self.conv_fuse(out)

        return out


# --- 2. C2f_EPSA Block Definition ---
class C2f_EPSA(nn.Module):
    """C2f block with Efficient Pyramid Squeeze Attention (EPSA/PSA) module."""
    def __init__(self, c1, c2, n=1, shortcut=False, g=1, e=0.5, scales=4, reduction=16):
        super().__init__()
        self.c = int(c2 * e)  # hidden channels
        self.cv1 = Conv(c1, 2 * self.c, 1, 1)
        self.cv2 = Conv((2 + n) * self.c, c2, 1)
        self.m = nn.ModuleList(
            nn.Sequential(
                Conv(self.c, self.c, 3, 1, g=g),
                Conv(self.c, self.c, 3, 1, g=g)
            ) for _ in range(n)
        )
        # 🟢 THE EPSA ATTENTION INJECTION 🟢
        # Initialize PSA_Attention module
        self.att = PSA_Attention(c2, scales=scales, reduction=reduction)

    def forward(self, x):
        # Standard C2f Logic
        y = list(self.cv1(x).chunk(2, 1))
        y.extend(m(y[-1]) for m in self.m)
        out = self.cv2(torch.cat(y, 1))

        # Apply EPSA Attention on the output features
        return self.att(out)

# --- 3. Model Injection Function ---
def create_epsa_yolo_model(model_path='yolov8n.pt'):
    """
    Loads a standard YOLO model and replaces C2f blocks
    with C2f_EPSA blocks in the backbone/head.
    """
    print(f"🏗 Loading base model: {model_path}...")
    # NOTE: The provided model_asr is no longer used, as we must load a fresh model
    # or ensure the base model is a standard one before modification.
    # Using 'yolov8n.pt' which is a standard base model.
    model = YOLO(model_path)

    print("🔁 Injecting Efficient Pyramid Squeeze Attention (EPSA) Modules into architecture...")

    nn_model = model.model

    for name, module in nn_model.named_modules():
        # Only target standard C2f blocks (which C2f_ASR inherits from structurally)
        if isinstance(module, C2f):
            # Get parent module and child name for replacement
            parent_name = name.rsplit('.', 1)[0] if '.' in name else ''
            child_name = name.rsplit('.', 1)[-1]

            parent = nn_model.get_submodule(parent_name) if parent_name else nn_model

            # Extract parameters from existing C2f
            c1 = module.cv1.conv.in_channels
            c2 = module.cv2.conv.out_channels
            n = len(module.m)

            # Create new EPSA C2f
            # NOTE: Default EPSA parameters (scales=4, reduction=16) are used.
            new_module = C2f_EPSA(c1, c2, n=n)

            # Replace
            setattr(parent, child_name, new_module)

    print("✅ Model transformation complete. EPSA Attention active.")
    return model

# Usage: Create the EPSA-enhanced model
model_epsa = create_epsa_yolo_model('yolov8n.pt')

print("\n🎉 STEP 4 COMPLETED: EPSA Attention Architecture ready.")

🏗 Loading base model: yolov8n.pt...
🔁 Injecting Efficient Pyramid Squeeze Attention (EPSA) Modules into architecture...
✅ Model transformation complete. EPSA Attention active.

🎉 STEP 4 COMPLETED: EPSA Attention Architecture ready.


/usr/local/lib/python3.12/dist-packages/torch/nn/init.py:566: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


In [ ]:
# 1️⃣ Remove any previously downloaded dataset directory to prevent conflicts
import shutil

old_dataset_path = '/content/datasets/brain_tumor_detection'
if os.path.exists(old_dataset_path):
    shutil.rmtree(old_dataset_path)
    print(f"Cleaned up old dataset directory: {old_dataset_path}")

# 2️⃣ Download the new dataset from Kaggle
print("Downloading new dataset from Kaggle: sartajbhuvaji/brain-tumor-classification-mri...")
!kaggle datasets download -d sartajbhuvaji/brain-tumor-classification-mri

# 3️⃣ Unzip the dataset to a temporary location
print("Unzipping dataset...")
new_zip_file = 'brain-tumor-classification-mri.zip'
tmp_extract_dir = '/tmp/figshare_dataset_download'
os.makedirs(tmp_extract_dir, exist_ok=True)
!unzip -q {new_zip_file} -d {tmp_extract_dir}

# 4️⃣ Define the target directory for the organized dataset
final_dataset_root = '/content/datasets/figshare_mri'
os.makedirs(final_dataset_root, exist_ok=True)

# 5️⃣ Move contents from the temporary extraction folder to the final dataset directory
# The dataset typically unzips into a folder like 'brain_tumor_dataset' which contains Training, Testing, Validation
# We need to find this intermediate folder if it exists, or assume direct extraction.

extracted_subfolders = [f.name for f in os.scandir(tmp_extract_dir) if f.is_dir()]

# Corrected logic to move the subfolders found directly in tmp_extract_dir
if len(extracted_subfolders) > 0:
    for folder_name in extracted_subfolders:
        source_path = os.path.join(tmp_extract_dir, folder_name)
        destination_path = os.path.join(final_dataset_root, folder_name)
        shutil.move(source_path, destination_path)
    print(f"Moved content from {tmp_extract_dir} to {final_dataset_root}")
else:
    print(f"No subfolders found in {tmp_extract_dir} to move.")

# 6️⃣ Clean up the downloaded zip file and temporary extraction folder
!rm {new_zip_file}
!rm -r {tmp_extract_dir}

print(f"✅ New dataset downloaded, unzipped, and reorganized to {final_dataset_root}")

Dataset URL: https://www.kaggle.com/datasets/sartajbhuvaji/brain-tumor-classification-mri
License(s): MIT
  0% 0.00/86.8M [00:00<?, ?B/s]
100% 86.8M/86.8M [00:00<00:00, 1.77GB/s]
Unzipping dataset...
Moved content from /tmp/figshare_dataset_download to /content/datasets/figshare_mri
✅ New dataset downloaded, unzipped, and reorganized to /content/datasets/figshare_mri


In [ ]:
import os
from pathlib import Path
from PIL import Image # Pillow for image size
import shutil # Import shutil for file operations
from sklearn.model_selection import train_test_split

print("🛠 Starting dataset conversion to YOLO format...")

source_dataset_root = '/content/datasets/figshare_mri' # Updated source path
target_dataset_root = '/content/datasets/figshare_mri_yolo' # Updated target path

# Class mapping for YOLO labels - UPDATED CLASS NAMES
class_names = ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']
class_to_id = {name: i for i, name in enumerate(class_names)}

# Create target directories
for split in ['train', 'val', 'test']:
    Path(f'{target_dataset_root}/{split}/images').mkdir(parents=True, exist_ok=True)
    Path(f'{target_dataset_root}/{split}/labels').mkdir(parents=True, exist_ok=True)

# Process Training split to create train and validation sets
training_source_path = Path(source_dataset_root) / 'Training'
if training_source_path.exists():
    print(f"Processing Training split for train/val...")
    all_training_images = []
    for class_folder in training_source_path.iterdir():
        if class_folder.is_dir():
            class_name = class_folder.name.lower()
            if class_to_id.get(class_name) is not None:
                all_training_images.extend(list(class_folder.glob('*.jpg')))

    # Split training images into train and validation
    train_images, val_images = train_test_split(all_training_images, test_size=0.2, random_state=42)

    # Copy images and create labels for the training set
    for img_file in train_images:
        class_name = img_file.parent.name.lower()
        class_id = class_to_id[class_name]
        shutil.copy(img_file, Path(target_dataset_root) / 'train' / 'images' / img_file.name)
        label_filename = img_file.stem + '.txt'
        with open(Path(target_dataset_root) / 'train' / 'labels' / label_filename, 'w') as f:
            f.write(f"{class_id} 0.5 0.5 1.0 1.0\n")

    # Copy images and create labels for the validation set
    for img_file in val_images:
        class_name = img_file.parent.name.lower()
        class_id = class_to_id[class_name]
        shutil.copy(img_file, Path(target_dataset_root) / 'val' / 'images' / img_file.name)
        label_filename = img_file.stem + '.txt'
        with open(Path(target_dataset_root) / 'val' / 'labels' / label_filename, 'w') as f:
            f.write(f"{class_id} 0.5 0.5 1.0 1.0\n")
else:
    print(f"⚠️  Warning: Original training folder '{training_source_path}' not found. No train/val split created.")

# Process Testing split for the test set
testing_source_path = Path(source_dataset_root) / 'Testing'
if testing_source_path.exists():
    print(f"Processing Testing split for test...")
    for class_folder in testing_source_path.iterdir():
        if class_folder.is_dir():
            class_name = class_folder.name.lower()
            class_id = class_to_id.get(class_name)

            if class_id is None:
                print(f"  Skipping unknown class folder: {class_name}")
                continue

            for img_file in class_folder.glob('*.jpg'):
                shutil.copy(img_file, Path(target_dataset_root) / 'test' / 'images' / img_file.name)
                label_filename = img_file.stem + '.txt'
                with open(Path(target_dataset_root) / 'test' / 'labels' / label_filename, 'w') as f:
                    f.write(f"{class_id} 0.5 0.5 1.0 1.0\n")
else:
    print(f"⚠️  Warning: Original testing folder '{testing_source_path}' not found. No test split created.")

print(f"✅ Dataset conversion complete. YOLO formatted data is at {target_dataset_root}")

🛠 Starting dataset conversion to YOLO format...
Processing Training split for train/val...
Processing Testing split for test...
✅ Dataset conversion complete. YOLO formatted data is at /content/datasets/figshare_mri_yolo


In [ ]:
# 3️⃣ Create the data.yaml file
# This file tells YOLO where to find the images, labels, and class names.

# Define paths and class names - UPDATED FOR NEW DATASET
dataset_path = '/content/datasets/figshare_mri_yolo' # Updated path for YOLO formatted data

# Define the content for data.yaml
data_yaml_content = {
    'path': dataset_path, # Dataset root directory
    'train': 'train/images', # Relative path for training images
    'val': 'val/images',   # Relative path for validation images (even if empty, for consistency)
    'test': 'test/images', # Relative path for test images (optional, for final evaluation)
    'nc': 4,                  # Number of classes
    'names': ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']  # Updated Class names
}

# Create the data.yaml file
os.makedirs('/content/datasets', exist_ok=True) # Ensure the directory exists
with open('/content/datasets/data.yaml', 'w') as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False)

print("✅ data.yaml created:")
with open('/content/datasets/data.yaml', 'r') as f:
    print(f.read())

print("✅ STEP 2 COMPLETED: Dataset prepared and data.yaml created.")

✅ data.yaml created:
names:
- glioma_tumor
- meningioma_tumor
- no_tumor
- pituitary_tumor
nc: 4
path: /content/datasets/figshare_mri_yolo
test: test/images
train: train/images
val: val/images

✅ STEP 2 COMPLETED: Dataset prepared and data.yaml created.


In [ ]:
import os
import torch
import torch.nn.functional as F
from ultralytics import YOLO

# --- (Assuming PSA_Attention, C2f_EPSA, and model_epsa are already defined and in scope) ---

# --- Fix 1: Manually set the model path in overrides ---
# This ensures the 'model' key exists and points to the original weights
# so the engine can initialize correctly.
model_epsa.overrides['model'] = 'yolov8n.pt'
print("🔧 Fixed: Injected 'yolov8n.pt' into model_epsa.overrides['model'].")

# --- Fix 2: Explicitly set the task ---
# Ensure the model task is correctly set for training
if 'task' not in model_epsa.overrides:
    model_epsa.overrides['task'] = model_epsa.task
    print(f"🔧 Fixed: Set model_epsa.overrides['task'] to {model_epsa.task}.")

# ----------------------------------------------------

print("🚀 Starting Training for EPSANet-enhanced YOLOv8...")

# 1. Check if data.yaml exists (Good practice)
yaml_path = '/content/datasets/data.yaml'
if not os.path.exists(yaml_path):
    raise FileNotFoundError(f"❌ data.yaml not found at {yaml_path}. Did you run Step 2?")

# 2. Run Training
# Note: We set resume=False for the first run on a modified model.
results_epsa = model_epsa.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=0,
    project='runs/train',
    name='epsa_yolo',
    exist_ok=True,
    optimizer='AdamW',
    lr0=0.001,
    amp=False,
    plots=True,
    resume=False # ❌ IMPORTANT CHANGE: Disable resume for the first run on the modified architecture!
)

print(f"✅ STEP 4 COMPLETED: Training finished for EPSANet-YOLOv8.")
print(f"   -> Best weights saved at: {results_epsa.save_dir}/weights/best.pt")
print(f"   -> Training graphs saved at: {results_epsa.save_dir}")

🔧 Fixed: Injected 'yolov8n.pt' into model_epsa.overrides['model'].
🚀 Starting Training for EPSANet-enhanced YOLOv8...
Ultralytics 8.3.233 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=False, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=epsa_yolo, nbs=64, n